In [ ]:
"""!pip install pillow
!pip install pandas
!pip install scipy
!pip install scikit-learn
!pip install kagglehub"""

'!pip install pillow\n!pip install pandas\n!pip install scipy\n!pip install scikit-learn\n!pip install kagglehub'

In [ ]:
"""!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121"""

'!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121'

In [ ]:
"""import os
import kagglehub

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import pandas as pd

# Download latest version
path = kagglehub.dataset_download("nicolacarrassi/ava-aesthetic-visual-assessment")

print("Path to dataset files:", path)

csv_path = os.path.join(path, "ground_truth_dataset.csv")
image_dir = os.path.join(path, "images")"""

'import os\nimport kagglehub\n\nfrom PIL import ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nimport pandas as pd\n\n# Download latest version\npath = kagglehub.dataset_download("nicolacarrassi/ava-aesthetic-visual-assessment")\n\nprint("Path to dataset files:", path)\n\ncsv_path = os.path.join(path, "ground_truth_dataset.csv")\nimage_dir = os.path.join(path, "images")'

In [ ]:
import os
import kagglehub

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import pandas as pd

path = '/home/elicer/.cache/kagglehub/datasets/nicolacarrassi/ava-aesthetic-visual-assessment/versions/1'

csv_path = os.path.join(path, "ground_truth_dataset.csv")
image_dir = os.path.join(path, "images")

/home/elicer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv(csv_path)

print(df.head(), flush=True)
print(df.columns, flush=True)

   image_num  vote_1    vote_2    vote_3    vote_4    vote_5    vote_6  \
0     953417     0.0  0.000000  0.000000  0.040323  0.258065  0.403226   
1     953777     0.0  0.023438  0.015625  0.023438  0.101562  0.312500   
2     953756     0.0  0.015625  0.023438  0.070312  0.273438  0.390625   
3     954195     0.0  0.008197  0.057377  0.213115  0.459016  0.188525   
4     953903     0.0  0.008065  0.032258  0.040323  0.266129  0.403226   

     vote_7    vote_8    vote_9   vote_10  
0  0.185484  0.080645  0.024194  0.008065  
1  0.273438  0.164062  0.062500  0.023438  
2  0.156250  0.039062  0.015625  0.015625  
3  0.049180  0.008197  0.000000  0.016393  
4  0.137097  0.072581  0.024194  0.016129  
Index(['image_num', 'vote_1', 'vote_2', 'vote_3', 'vote_4', 'vote_5', 'vote_6',
       'vote_7', 'vote_8', 'vote_9', 'vote_10'],
      dtype='object')


In [ ]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)
print(torch.__file__)
print(torchvision.__file__)

2.5.1+cu121
0.20.1+cu121
/home/elicer/.local/lib/python3.10/site-packages/torch/__init__.py
/home/elicer/.local/lib/python3.10/site-packages/torchvision/__init__.py


In [ ]:
import os
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class AVADataset(Dataset):
    VOTE_COLS = [
        'vote_1', 'vote_2', 'vote_3', 'vote_4', 'vote_5',
        'vote_6', 'vote_7', 'vote_8', 'vote_9', 'vote_10'
    ]

    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = row['image_num']

        label = torch.tensor(
            row[self.VOTE_COLS].values.astype('float32')
        )
        label = label / label.sum().clamp_min(1e-6)

        image_path = os.path.join(
            self.image_dir,
            f"{int(image_id)}.jpg"
        )

        try:
            if not os.path.exists(image_path):
                raise FileNotFoundError(f"Missing: {image_path}")

            image = Image.open(image_path).convert("RGB")

            if self.transform:
                image = self.transform(image)

        except (OSError, FileNotFoundError, Exception) as e:
            print(f"Skipping corrupted image {image_id}: {e}")
            return self.__getitem__((idx + 1) % len(self.df))

        return image, label

In [ ]:
import torch.nn as nn
from torchvision import models

weights = models.Swin_T_Weights.IMAGENET1K_V1

model = models.swin_t(weights=weights)

model.head = nn.Linear(model.head.in_features, 10)

for p in model.parameters():
    p.requires_grad = False

for p in model.head.parameters():
    p.requires_grad = True

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df), flush=True)

204406 25551 25551


In [ ]:
train_dataset = AVADataset(
    train_df,
    image_dir,
    transform=train_transform
)

val_dataset = AVADataset(
    val_df,
    image_dir,
    transform=eval_transform
)

test_dataset = AVADataset(
    test_df,
    image_dir,
    transform=eval_transform
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

/home/elicer/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
images, labels = next(iter(train_loader))

print(images.shape, flush=True)
print(labels.shape, flush=True)
print(labels[0].sum(), flush=True)

torch.Size([64, 3, 224, 224])
torch.Size([64, 10])
tensor(1.)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EMDLoss(nn.Module):
    def __init__(self, r=2):
        super(EMDLoss, self).__init__()
        self.r = r

    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1)

        cdf_pred = torch.cumsum(pred, dim=1)
        cdf_target = torch.cumsum(target, dim=1)

        samplewise_emd = torch.mean(
            torch.abs(cdf_pred - cdf_target) ** self.r,
            dim=1
        )

        return torch.mean(samplewise_emd)

In [ ]:
import torch

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(torch.cuda.is_available())

model = model.to(device)

print_batch = 100

num_epochs = 100
learning_rate = 1e-4
learning_rate_after_unfreeze = 1e-5

criterion = EMDLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

scaler = torch.cuda.amp.GradScaler()

first_unfreeze_epoch = 3
second_unfreeze_epoch = 6

def evaluate(data_loader):
    model.eval()
    val_loss_sum = 0.0
    val_mae_sum = 0.0
    num_samples = 0

    scores = torch.arange(1, 11, device=device, dtype=torch.float32)

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            cur_batch_size = images.size(0)
            num_samples += cur_batch_size

            val_loss_sum += loss.item() * cur_batch_size

            pred_dist = torch.softmax(outputs.float(), dim=1)
            pred_mean = (pred_dist * scores).sum(dim=1)
            true_mean = (labels * scores).sum(dim=1)

            mae = torch.abs(pred_mean - true_mean).sum()
            val_mae_sum += mae.item()

    return val_loss_sum / num_samples, val_mae_sum / num_samples


total_step = len(train_loader)

best_val_mae = float('inf')
patience = 3
early_stop_counter = 0

print("Starting training...", flush=True)

for epoch in range(num_epochs):

    if epoch == first_unfreeze_epoch:
        print("Unfreezing Swin_T last block", flush=True)

        for p in model.features[-1].parameters():
          p.requires_grad = True

        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=learning_rate_after_unfreeze
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=2
        )


    model.train()
    train_loss_sum = 0.0
    num_samples = 0

    for batch_index, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        cur_batch_size = images.size(0)
        train_loss_sum += loss.item() * cur_batch_size
        num_samples += cur_batch_size

        if (batch_index + 1) % print_batch == 0:
            train_loss = train_loss_sum / num_samples
            print(
                "Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}".format(
                    epoch + 1,
                    num_epochs,
                    batch_index + 1,
                    total_step,
                    train_loss
                ),
                flush=True
            )

    val_loss, val_mae = evaluate(val_loader)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] '
        f'Validation Loss: {val_loss:.4f} '
        f'Validation MAE: {val_mae:.4f}',
        flush=True
    )

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        early_stop_counter = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_mae": best_val_mae,
        }, "./Swin_T_best.pt")

        print("Best model saved.", flush=True)

    else:
        early_stop_counter += 1
        print(f"Early stop counter: {early_stop_counter}/{patience}", flush=True)

    scheduler.step(val_mae)

    if early_stop_counter >= patience:
        print("Early Stopping!", flush=True)
        break


checkpoint = torch.load("./Swin_T_best.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

_, test_mae = evaluate(test_loader)
print('MAE value of the model on the test images: {}'.format(test_mae), flush=True)

In [ ]:
from scipy.stats import pearsonr, spearmanr

def calculate_all_metrics(y_true_dist, y_pred_dist):
    scores = torch.arange(1, 11).float().to(y_pred_dist.device)

    true_means = torch.sum(y_true_dist * scores, dim=1)
    pred_means = torch.sum(y_pred_dist * scores, dim=1)

    true_std = torch.sqrt(torch.sum(y_true_dist * (scores**2), dim=1) - true_means**2)
    pred_std = torch.sqrt(torch.sum(y_pred_dist * (scores**2), dim=1) - pred_means**2)

    lcc_mean, _ = pearsonr(pred_means.cpu().numpy(), true_means.cpu().numpy())
    srcc_mean, _ = spearmanr(pred_means.cpu().numpy(), true_means.cpu().numpy())

    lcc_std, _ = pearsonr(pred_std.cpu().numpy(), true_std.cpu().numpy())
    srcc_std, _ = spearmanr(pred_std.cpu().numpy(), true_std.cpu().numpy())

    true_labels = (true_means >= 5.0).float()
    pred_labels = (pred_means >= 5.0).float()
    accuracy = (true_labels == pred_labels).float().mean().item()

    cdf_true = torch.cumsum(y_true_dist, dim=1)
    cdf_pred = torch.cumsum(y_pred_dist, dim=1)
    emd = torch.norm(cdf_true - cdf_pred, p=1, dim=1).mean().item() / 10

    return {
        "Accuracy": accuracy * 100,
        "LCC_mean": lcc_mean,
        "SRCC_mean": srcc_mean,
        "LCC_std": lcc_std,
        "SRCC_std": srcc_std,
        "EMD": emd
    }


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('./Swin_T_best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

all_metrics = []

with torch.no_grad():
    for images, target_dist in val_loader:
        images = images.to(device)
        target_dist = target_dist.to(device)

        logits = model(images)

        pred_dist = F.softmax(logits, dim=1)

        batch_metrics = calculate_all_metrics(target_dist, pred_dist)
        all_metrics.append(batch_metrics)

final_results = {}
for key in all_metrics[0].keys():
    final_results[key] = np.mean([m[key] for m in all_metrics])

print("--- 최종 테스트 결과 ---")
for key, value in final_results.items():
    print(f"{key}: {value:.4f}")

/tmp/ipykernel_71/783908181.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./vit_best.pt', map_location=device)
/home/elicer/.local/lib/python3

--- 최종 테스트 결과 ---
Accuracy: 78.8935
LCC_mean: 0.6627
SRCC_mean: 0.6441
LCC_std: 0.2895
SRCC_std: 0.2714
EMD: 0.0480
